# UNSW-NB15 Label Harmonization  
*@Ullas*

Builds a deterministic `attack_cat → int` mapping that every downstream module
(multiclass training, inference, API) will use.  
Output: `models/unsw_label_mapping.json`

In [ ]:
import os
import glob
import json
import pandas as pd

# Resolve repo root regardless of working directory
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
DATA_DIR   = os.path.join(REPO_ROOT, "data", "UNSW-NB15")
MODELS_DIR = os.path.join(REPO_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"MODELS_DIR : {MODELS_DIR}")

## 1 — Load Training Data

Prefer `UNSW_NB15_training-set.csv` if present; otherwise concatenate the four
raw partitions `UNSW-NB15_1.csv … UNSW-NB15_4.csv`.  
We only need the `attack_cat` and `label` columns for this notebook.

In [ ]:
# UNSW-NB15 CSV column names (the raw files have no header row)
UNSW_COLS = [
    "srcip", "sport", "dstip", "dsport", "proto", "state", "dur",
    "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "service",
    "sload", "dload", "spkts", "dpkts", "swin", "dwin", "stcpb", "dtcpb",
    "smeansz", "dmeansz", "trans_depth", "res_bdy_len", "sjit", "djit",
    "stime", "ltime", "sintpkt", "dintpkt", "tcprtt", "synack", "ackdat",
    "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login",
    "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_dport_ltm",
    "ct_dst_sport_ltm", "ct_dst_src_ltm", "ct_src_ltm", "attack_cat", "label"
]

# Try training-set CSV first, then fall back to raw partitions
training_set_path = os.path.join(DATA_DIR, "UNSW_NB15_training-set.csv")

if os.path.exists(training_set_path):
    print("Loading UNSW_NB15_training-set.csv …")
    df = pd.read_csv(training_set_path)
else:
    # Raw partition files (no header)
    raw_paths = sorted(glob.glob(os.path.join(DATA_DIR, "UNSW-NB15_*.csv")))
    if not raw_paths:
        raise FileNotFoundError(
            f"No UNSW-NB15 CSV files found in {DATA_DIR}. "
            "Download them from https://research.unsw.edu.au/projects/unsw-nb15-dataset"
        )
    print(f"Loading {len(raw_paths)} raw CSV partition(s) …")
    parts = [pd.read_csv(p, header=None, names=UNSW_COLS, low_memory=False) for p in raw_paths]
    df = pd.concat(parts, ignore_index=True)

print(f"\nShape          : {df.shape}")
print(f"Columns present: {list(df.columns[:10])} …")
print(f"\nLabel dist:\n{df['label'].value_counts().to_string()}")
df[["attack_cat", "label"]].head(10)

## 2 — Explore Attack Categories

Check unique values, value counts, nulls, and whitespace inconsistencies.

In [ ]:
# Normalize: strip whitespace, unify casing (title-case)
df["attack_cat"] = df["attack_cat"].astype(str).str.strip()
# Replace empty / nan strings that slipped in as text
df["attack_cat"] = df["attack_cat"].replace({"nan": "Normal", "": "Normal"})

null_count = df["attack_cat"].isnull().sum()
print(f"Null attack_cat values : {null_count}")

vc = df["attack_cat"].value_counts()
print(f"\nattack_cat value counts ({len(vc)} unique):\n{vc.to_string()}")

print(f"\nUnique attack_cat values (raw):\n{sorted(df['attack_cat'].unique())}")